In [12]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# PERCEIVED — exactly as YouGov conjoint had it. Do not reorder.
perceived = ["Debt interest","NHS","Working age benefits","Public order & safety",
    "Defence","Pensioners","Education","Social care","Overseas aid","Housing","Transport"]
# ACTUAL — true PESA 2024-25 order (£bn): debt interest = #4
actual = ["Working age benefits","NHS","Pensioners","Debt interest","Education",
    "Defence","Public order & safety","Transport","Social care","Housing","Overseas aid"]

rows = []
for r,c in enumerate(perceived, 1): rows.append(("Perceived", c, r))
for r,c in enumerate(actual, 1):    rows.append(("Actual", c, r))
df = pd.DataFrame(rows, columns=["side","cat","rank"])

x = alt.X("side:N", sort=["Perceived","Actual"], title=None,
          axis=alt.Axis(labels=False, ticks=False, domain=False))
y = alt.Y("rank:Q", scale=alt.Scale(reverse=True, domain=[0.5, 11.5]), axis=None)
col = alt.Color("cat:N", scale=alt.Scale(scheme="category20"), legend=None)

lines = alt.Chart(df).mark_line(interpolate="monotone", strokeWidth=3).encode(
    x=x, y=y, detail="cat:N", color=col)
pts = alt.Chart(df).mark_point(filled=True, size=90).encode(x=x, y=y, color=col)

di = df[df.cat=="Debt interest"]
di_line = alt.Chart(di).mark_line(interpolate="monotone", strokeWidth=5.5).encode(
    x=x, y=y, detail="cat:N", color=col)
di_pts = alt.Chart(di).mark_point(filled=True, size=140).encode(x=x, y=y, color=col)

p = df[df.side=="Perceived"].copy(); p["txt"] = p["rank"].astype(str)+"   "+p["cat"]
a = df[df.side=="Actual"].copy();    a["txt"] = a["rank"].astype(str)+"   "+a["cat"]

lab_p = alt.Chart(p[p.cat!="Debt interest"]).mark_text(
    align="right", dx=-16, fontSize=12, color="#122b39").encode(x=x, y=y, text="txt:N")
lab_a = alt.Chart(a[a.cat!="Debt interest"]).mark_text(
    align="left", dx=16, fontSize=12, color="#122b39").encode(x=x, y=y, text="txt:N")
lab_pd = alt.Chart(p[p.cat=="Debt interest"]).mark_text(
    align="right", dx=-16, fontSize=12, fontWeight="bold", color="#122b39").encode(x=x, y=y, text="txt:N")
lab_ad = alt.Chart(a[a.cat=="Debt interest"]).mark_text(
    align="left", dx=16, fontSize=12, fontWeight="bold", color="#122b39").encode(x=x, y=y, text="txt:N")

h_p = alt.Chart(pd.DataFrame([{"side":"Perceived","t":"What the public thinks"}])).mark_text(
    fontSize=13, fontWeight="bold", color="#122b39", align="right", dx=-16, dy=-10).encode(
    x=x, y=alt.value(0), text="t:N")
h_a = alt.Chart(pd.DataFrame([{"side":"Actual","t":"What's actually spent"}])).mark_text(
    fontSize=13, fontWeight="bold", color="#122b39", align="left", dx=16, dy=-10).encode(
    x=x, y=alt.value(0), text="t:N")

caption = alt.Title(
    text="Source: YouGov (perceived, Nov 2025, n=4,197); HMRC / PESA 2024–25 (actual)",
    subtitle=["Debt interest is ranked the #1 cost by the public — in reality, the fourth-largest."],
    orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (
    (lines + pts + di_line + di_pts + lab_p + lab_a + lab_pd + lab_ad + h_p + h_a)
    .properties(width=300, height=270,
                title=caption)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

styles.save(chart, name="misperception_gap", svg=True)
chart


alt.LayerChart(...)